In [1]:
# %pip install transformers datasets accelerate sentencepiece
# %pip install datasets==2.17.1
# %pip install transformers==4.40.0
# %pip install numpy==1.26.4 --force-reinstall




In [ ]:
# %pip install --upgrade transformers==4.41.2
# %pip install --upgrade transformers==4.46.1

# %pip install modernbert


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement modernbert (from versions: none)
ERROR: No matching distribution found for modernbert


In [3]:
import pandas as pd
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

d:\LPA_MTech_Project\lpavenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np
print(np.__version__)


1.26.4


In [5]:
start = int(input("Enter Start Year: "))
end = int(input("Enter Ending Year: "))


In [6]:
# Load parquet
df = pd.read_parquet(f"D:\LPA_MTech_Project\Enriched_Datasets\SupremeCourt_Combined_{start}_{end}_enriched.parquet")

# Keep only needed columns
df = df[["text", "verdict_label"]]
df = df.rename(columns={"verdict_label": "label"})

# Drop rows missing text/labels
df = df.dropna(subset=["text", "label"])

# Convert to HF Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.15, seed=42)

In [ ]:

# model_name = "answerdotai/ModernBERT-base"

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model = AutoModelForSequenceClassification.from_pretrained(
#     model_name,
#     num_labels=2   # binary classification
# )

from modernbert import ModernBertForSequenceClassification
from transformers import AutoTokenizer

model_name = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = ModernBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("ModernBERT loaded!")


ValueError: The checkpoint you are trying to load has model type `modernbert` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        max_length=2048,         # SAFE for GTX1650 (4GB)
        truncation=True,
        padding="max_length"
    )

tokenized = dataset.map(tokenize, batched=True, batch_size=16)
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format("torch")


In [ ]:
training_args = TrainingArguments(
    output_dir="modernbert_verdict",
    num_train_epochs=4,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    warmup_steps=200,
    learning_rate=2e-5,
    weight_decay=0.01,

    fp16=True,
    optim="adamw_torch_fused",

    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",

    gradient_checkpointing=True,
    dataloader_num_workers=0,   # <-- FIX HERE
)


In [ ]:
from transformers import TrainingArguments
import inspect
print(TrainingArguments.__module__)
print(inspect.getsourcefile(TrainingArguments))


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
)


In [ ]:
# Check 1: Is CUDA available?
print("CUDA Available:", torch.cuda.is_available())

# Check 2: Which GPU is detected?
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Device Count:", torch.cuda.device_count())

# Check 3: Confirm model is on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model is on device:", next(model.parameters()).device)

In [ ]:
trainer.train()


In [ ]:
print(trainer.evaluate())
